# Martingale Staking — Drawdown & Streak Analysis

Simulates two-team sports markets and evaluates a **Martingale recoup staking system** under two strategies:

- **Favourites agent** — always bets on the team with lower odds (higher implied probability)
- **Outsiders agent** — always bets on the team with higher odds (lower implied probability)

### Staking rule
- First bet (and any bet after a win): **\$1 stake**
- After a loss: stake enough to **recoup all accumulated losses + \$1 profit** at the current market's odds
  - `stake = (cumulative_losses + 1) / ((odds - 1) × (1 - commission))`
- A win resets the stake sequence back to \$1

### Key questions
1. What drawdowns should you expect, and how bad can they get?
2. How long are typical losing streaks, and what is the worst-case?
3. How much starting bankroll do you actually need?
4. Does betting favourites vs. outsiders change the risk profile meaningfully?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec

rng = np.random.default_rng(seed=42)

# ── Simulation parameters ──────────────────────────────────────────────────
N_BETS          = 10_000    # bets per simulation
N_SIMS          = 1_000     # Monte Carlo runs
STARTING_BAL    = 10_000.0  # starting bankroll ($)
OVERROUND       = 1.05      # 105% book
COMMISSION      = 0.05      # 5% commission on winning profit
MIN_TRUE_PROB   = 0.30      # narrowest favourite
MAX_TRUE_PROB   = 0.70      # widest favourite

FAVOURITE_COLOR = '#2196F3'
OUTSIDER_COLOR  = '#FF5722'

## Simulation engine

In [ ]:
def simulate_martingale(
    agent_type: str,
    n_bets: int = N_BETS,
    starting_balance: float = STARTING_BAL,
    overround: float = OVERROUND,
    commission: float = COMMISSION,
    min_true_prob: float = MIN_TRUE_PROB,
    max_true_prob: float = MAX_TRUE_PROB,
    local_rng=None,
) -> dict:
    """
    Run one Martingale simulation.

    Parameters
    ----------
    agent_type : 'favourite' or 'outsider'

    Returns
    -------
    dict with:
        final_balance, max_drawdown, max_win_streak, max_loss_streak,
        max_stake, bust (bool), bust_step, balance_history (np.ndarray)
    """
    if local_rng is None:
        local_rng = rng

    # Pre-generate all markets for this simulation
    true_prob_a = local_rng.uniform(min_true_prob, max_true_prob, size=n_bets)
    odds_a = 1.0 / (true_prob_a * overround)
    odds_b = 1.0 / ((1.0 - true_prob_a) * overround)
    team_a_wins = local_rng.random(size=n_bets) < true_prob_a

    balance          = starting_balance
    cumulative_loss  = 0.0
    peak_balance     = starting_balance
    max_drawdown     = 0.0
    max_stake        = 0.0
    cur_win_streak   = 0
    cur_loss_streak  = 0
    max_win_streak   = 0
    max_loss_streak  = 0
    bust             = False
    bust_step        = None

    balance_hist = np.empty(n_bets + 1)
    balance_hist[0] = balance

    steps_completed = 0

    for i in range(n_bets):
        # ── Pick team ──────────────────────────────────────────────────
        if agent_type == 'favourite':
            bet_on_a = odds_a[i] <= odds_b[i]   # shorter odds = favourite
        else:
            bet_on_a = odds_a[i] > odds_b[i]    # longer odds = outsider

        bet_odds = odds_a[i] if bet_on_a else odds_b[i]

        # ── Martingale stake ───────────────────────────────────────────
        if cumulative_loss == 0.0:
            stake = 1.0
        else:
            net_profit_factor = (bet_odds - 1.0) * (1.0 - commission)
            stake = (cumulative_loss + 1.0) / net_profit_factor

        # ── Bust check: can't cover the required stake ─────────────────
        if stake > balance:
            bust      = True
            bust_step = i
            break

        max_stake = max(max_stake, stake)

        # ── Resolve ────────────────────────────────────────────────────
        won = (bet_on_a == team_a_wins[i])

        if won:
            net_profit      = stake * (bet_odds - 1.0) * (1.0 - commission)
            balance        += net_profit
            cumulative_loss = 0.0
            cur_win_streak += 1
            cur_loss_streak = 0
            max_win_streak  = max(max_win_streak, cur_win_streak)
        else:
            balance        -= stake
            cumulative_loss += stake
            cur_loss_streak += 1
            cur_win_streak   = 0
            max_loss_streak  = max(max_loss_streak, cur_loss_streak)

        peak_balance = max(peak_balance, balance)
        max_drawdown = max(max_drawdown, peak_balance - balance)

        steps_completed       += 1
        balance_hist[i + 1]    = balance

    return {
        'final_balance'  : balance,
        'max_drawdown'   : max_drawdown,
        'max_win_streak' : max_win_streak,
        'max_loss_streak': max_loss_streak,
        'max_stake'      : max_stake,
        'bust'           : bust,
        'bust_step'      : bust_step if bust else n_bets,
        'balance_history': balance_hist[: steps_completed + 1],
    }

## Single-run walkthrough
One representative simulation for each agent to get a feel for the dynamics.

In [ ]:
demo_rng = np.random.default_rng(seed=7)
fav_demo = simulate_martingale('favourite', local_rng=demo_rng)
demo_rng = np.random.default_rng(seed=7)   # same markets
out_demo = simulate_martingale('outsider',  local_rng=demo_rng)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

for ax, res, label, color in [
    (axes[0], fav_demo, 'Favourites', FAVOURITE_COLOR),
    (axes[1], out_demo, 'Outsiders',  OUTSIDER_COLOR),
]:
    hist = res['balance_history']
    x    = np.arange(len(hist))
    ax.plot(x, hist, color=color, lw=0.8, alpha=0.9)
    ax.axhline(STARTING_BAL, color='grey', ls='--', lw=0.8, label='Starting balance')

    # Shade drawdown area
    running_peak = np.maximum.accumulate(hist)
    ax.fill_between(x, hist, running_peak, alpha=0.15, color='red', label='Drawdown')

    bust_tag = f"  ← BUST at step {res['bust_step']:,}" if res['bust'] else ''
    ax.set_title(
        f"{label} agent — "
        f"Max DD: ${res['max_drawdown']:,.0f}  "
        f"Max loss streak: {res['max_loss_streak']}  "
        f"Max stake: ${res['max_stake']:,.2f}  "
        f"Final: ${res['final_balance']:,.0f}{bust_tag}",
        fontsize=10,
    )
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
    ax.set_xlabel('Bet number')
    ax.set_ylabel('Balance')
    ax.legend(fontsize=8)

fig.suptitle('Single-run Martingale — Same Markets, Different Selection Strategy', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Monte Carlo — run 1,000 simulations per agent
Each simulation uses independent market draws.

In [ ]:
def run_monte_carlo(agent_type: str, n_sims: int = N_SIMS, seed: int = 0) -> pd.DataFrame:
    results = []
    for i in range(n_sims):
        sim_rng = np.random.default_rng(seed + i)
        r = simulate_martingale(agent_type, local_rng=sim_rng)
        results.append({
            'final_balance'  : r['final_balance'],
            'max_drawdown'   : r['max_drawdown'],
            'max_win_streak' : r['max_win_streak'],
            'max_loss_streak': r['max_loss_streak'],
            'max_stake'      : r['max_stake'],
            'bust'           : r['bust'],
            'bust_step'      : r['bust_step'],
        })
    return pd.DataFrame(results)

print(f'Running {N_SIMS:,} simulations per agent ({N_BETS:,} bets each)...')
fav_df = run_monte_carlo('favourite', seed=1000)
out_df = run_monte_carlo('outsider',  seed=2000)
print('Done.')

## Summary statistics

In [ ]:
def summary_table(df: pd.DataFrame, label: str) -> pd.DataFrame:
    bust_rate = df['bust'].mean() * 100
    survived  = df[~df['bust']]

    rows = {}
    for col, fmt, name in [
        ('final_balance',   '${:,.0f}',  'Final balance'),
        ('max_drawdown',    '${:,.0f}',  'Max drawdown'),
        ('max_stake',       '${:,.2f}',  'Max single stake'),
        ('max_loss_streak', '{:.1f}',    'Max loss streak'),
        ('max_win_streak',  '{:.1f}',    'Max win streak'),
    ]:
        rows[name] = {
            'Mean'  : fmt.format(df[col].mean()),
            'Median': fmt.format(df[col].median()),
            'Std'   : fmt.format(df[col].std()),
            'Min'   : fmt.format(df[col].min()),
            'Max'   : fmt.format(df[col].max()),
            'p95'   : fmt.format(df[col].quantile(0.95)),
        }

    rows['Bust rate'] = {
        'Mean': f'{bust_rate:.1f}%', 'Median': '—', 'Std': '—',
        'Min': '—', 'Max': '—', 'p95': '—',
    }
    if bust_rate > 0:
        rows['Avg bust step'] = {
            'Mean'  : f"{df[df['bust']]['bust_step'].mean():,.0f}",
            'Median': f"{df[df['bust']]['bust_step'].median():,.0f}",
            'Std'   : '—', 'Min': '—', 'Max': '—', 'p95': '—',
        }

    return pd.DataFrame(rows).T

print('=== FAVOURITES AGENT ===')
display(summary_table(fav_df, 'Favourites'))
print('\n=== OUTSIDERS AGENT ===')
display(summary_table(out_df, 'Outsiders'))

## Drawdown distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Max drawdown ──────────────────────────────────────────────────────────
ax = axes[0]
for df, label, color in [
    (fav_df, 'Favourites', FAVOURITE_COLOR),
    (out_df, 'Outsiders',  OUTSIDER_COLOR),
]:
    ax.hist(
        df['max_drawdown'],
        bins=60, alpha=0.55, color=color, label=label, density=True, edgecolor='none',
    )
    ax.axvline(df['max_drawdown'].median(), color=color, ls='--', lw=1.5,
               label=f'{label} median: ${df["max_drawdown"].median():,.0f}')

ax.set_title('Max Drawdown Distribution', fontweight='bold')
ax.set_xlabel('Max Drawdown ($)')
ax.set_ylabel('Density')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
ax.legend(fontsize=8)

# ── Max stake ────────────────────────────────────────────────────────────
ax = axes[1]
for df, label, color in [
    (fav_df, 'Favourites', FAVOURITE_COLOR),
    (out_df, 'Outsiders',  OUTSIDER_COLOR),
]:
    ax.hist(
        df['max_stake'],
        bins=60, alpha=0.55, color=color, label=label, density=True, edgecolor='none',
    )
    ax.axvline(df['max_stake'].median(), color=color, ls='--', lw=1.5,
               label=f'{label} median: ${df["max_stake"].median():,.2f}')

ax.set_title('Max Single Stake Required', fontweight='bold')
ax.set_xlabel('Max Stake ($)')
ax.set_ylabel('Density')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
ax.legend(fontsize=8)

fig.suptitle(f'Drawdown Risk — {N_SIMS:,} Simulations × {N_BETS:,} Bets', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Streak distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in [
    (axes[0], 'max_loss_streak', 'Max Consecutive Losses'),
    (axes[1], 'max_win_streak',  'Max Consecutive Wins'),
]:
    for df, label, color in [
        (fav_df, 'Favourites', FAVOURITE_COLOR),
        (out_df, 'Outsiders',  OUTSIDER_COLOR),
    ]:
        vals    = df[col]
        bins    = np.arange(vals.min(), vals.max() + 2) - 0.5
        ax.hist(
            vals, bins=bins, alpha=0.55, color=color, label=label,
            density=True, edgecolor='none',
        )
        ax.axvline(vals.median(), color=color, ls='--', lw=1.5,
                   label=f'{label} median: {vals.median():.0f}')

    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Streak length (bets)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

fig.suptitle(f'Win / Loss Streak Distributions — {N_SIMS:,} Simulations', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Final balance distributions (survived runs only)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, label, color in [
    (axes[0], fav_df, 'Favourites', FAVOURITE_COLOR),
    (axes[1], out_df, 'Outsiders',  OUTSIDER_COLOR),
]:
    survived = df[~df['bust']]['final_balance']
    busted   = df[df['bust']]
    bust_pct = len(busted) / len(df) * 100

    ax.hist(survived, bins=50, color=color, alpha=0.7, edgecolor='none')
    ax.axvline(STARTING_BAL,       color='grey',  ls='--', lw=1.2, label='Starting balance')
    ax.axvline(survived.median(),  color='black', ls='-',  lw=1.5,
               label=f'Median: ${survived.median():,.0f}')

    ax.set_title(
        f'{label} — Survived runs ({100-bust_pct:.1f}%)\n'
        f'Bust rate: {bust_pct:.1f}%  |  Starting balance: ${STARTING_BAL:,.0f}',
        fontweight='bold', fontsize=9,
    )
    ax.set_xlabel('Final Balance ($)')
    ax.set_ylabel('Count')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
    ax.legend(fontsize=8)

fig.suptitle(f'Final Balance after {N_BETS:,} Bets — Survived Simulations Only', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Required bankroll to survive various percentiles

The Martingale system requires enough bankroll to cover the deepest losing run you'll encounter.
The table below shows the starting balance you'd need to survive the worst drawdown at each percentile.

In [ ]:
percentiles = [50, 75, 90, 95, 99]

rows = []
for p in percentiles:
    fav_dd = np.percentile(fav_df['max_drawdown'], p)
    out_dd = np.percentile(out_df['max_drawdown'], p)
    fav_sk = np.percentile(fav_df['max_stake'],    p)
    out_sk = np.percentile(out_df['max_stake'],    p)
    rows.append({
        'Percentile'                    : f'p{p}',
        'Fav — Max Drawdown'            : f'${fav_dd:>10,.0f}',
        'Out — Max Drawdown'            : f'${out_dd:>10,.0f}',
        'Fav — Max Single Stake'        : f'${fav_sk:>10,.2f}',
        'Out — Max Single Stake'        : f'${out_sk:>10,.2f}',
    })

req_df = pd.DataFrame(rows).set_index('Percentile')
display(req_df)

print(
    f"\nFav bust rate : {fav_df['bust'].mean()*100:.1f}%  "
    f"(of {N_SIMS} sims with ${STARTING_BAL:,.0f} starting bankroll)"
)
print(
    f"Out bust rate : {out_df['bust'].mean()*100:.1f}%  "
    f"(of {N_SIMS} sims with ${STARTING_BAL:,.0f} starting bankroll)"
)

## Head-to-head comparison chart

In [ ]:
fig = plt.figure(figsize=(14, 10))
gs  = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

metrics = [
    ('max_drawdown',    'Max Drawdown ($)',        True),
    ('max_stake',       'Max Single Stake ($)',    True),
    ('max_loss_streak', 'Max Loss Streak (bets)',  False),
    ('max_win_streak',  'Max Win Streak (bets)',   False),
    ('final_balance',   'Final Balance ($)',        True),
]

for idx, (col, ylabel, dollar) in enumerate(metrics):
    row, c = divmod(idx, 3)
    ax = fig.add_subplot(gs[row, c])

    fav_vals = fav_df[col]
    out_vals = out_df[col]
    combined = np.concatenate([fav_vals, out_vals])
    bins = np.linspace(combined.min(), np.percentile(combined, 98), 50)

    ax.hist(fav_vals, bins=bins, alpha=0.55, color=FAVOURITE_COLOR,
            label='Favourites', density=True, edgecolor='none')
    ax.hist(out_vals, bins=bins, alpha=0.55, color=OUTSIDER_COLOR,
            label='Outsiders',  density=True, edgecolor='none')
    ax.axvline(fav_vals.median(), color=FAVOURITE_COLOR, ls='--', lw=1.5)
    ax.axvline(out_vals.median(), color=OUTSIDER_COLOR,  ls='--', lw=1.5)

    ax.set_title(ylabel, fontsize=9, fontweight='bold')
    ax.set_ylabel('Density', fontsize=8)
    if dollar:
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
    ax.tick_params(axis='x', labelsize=7)
    ax.legend(fontsize=7)

# 6th panel: bust rate bar chart
ax = fig.add_subplot(gs[1, 2])
bust_rates = [fav_df['bust'].mean() * 100, out_df['bust'].mean() * 100]
bars = ax.bar(['Favourites', 'Outsiders'], bust_rates,
              color=[FAVOURITE_COLOR, OUTSIDER_COLOR], alpha=0.75, width=0.5)
for bar, rate in zip(bars, bust_rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{rate:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)
ax.set_title('Bust Rate (%)', fontsize=9, fontweight='bold')
ax.set_ylabel('% of simulations', fontsize=8)
ax.set_ylim(0, max(bust_rates) * 1.3 + 2)

fig.suptitle(
    f'Martingale System — Full Comparison\n'
    f'{N_SIMS:,} simulations × {N_BETS:,} bets  |  '
    f'Starting bankroll: ${STARTING_BAL:,.0f}  |  '
    f'Overround: {OVERROUND*100:.0f}%  |  Commission: {COMMISSION*100:.0f}%',
    fontsize=11, fontweight='bold',
)
plt.show()

---
## Sweet spot analysis — returns by odds bracket

Sweeps a fixed odds value across the full range of **favourites ($1.01 – $1.99)** and
**outsiders ($2.00 – $15.00)**, running 500 vectorised simulations per odds level.

Key question: *at which odds does Martingale minimise bust risk while preserving capital?*

**Counterintuitive finding:** very short-odds favourites (e.g. $1.05) bust *more* often than
longer odds, not less. The reason is stake escalation — when the profit margin is tiny, even
two or three consecutive losses force enormous recoup stakes. A $1 loss at odds $1.10
(net profit factor ≈ 0.095) requires a $21 stake next bet just to clear $2 profit. The same
loss at odds $1.80 (npf ≈ 0.76) only requires a $2.63 stake.

The sweet-spot is the odds bracket where survival probability and capital retention peak
simultaneously.

In [ ]:
# ── Vectorised fixed-odds sweep ───────────────────────────────────────────────
# Simulates n_sims paths in parallel for each fixed odds value using numpy,
# making the full sweep feasible within a notebook.

SWEEP_N_SIMS = 500
SWEEP_N_BETS = 10_000

def sweep_fixed_odds(
    odds_array,
    n_sims: int = SWEEP_N_SIMS,
    n_bets: int = SWEEP_N_BETS,
    starting_balance: float = STARTING_BAL,
    commission: float = COMMISSION,
    overround: float = OVERROUND,
    seed: int = 0,
) -> pd.DataFrame:
    """
    For each odds value in odds_array, simulate n_sims Martingale paths in
    parallel using pre-generated numpy arrays.

    true_prob per bet = 1 / (odds × overround)  — consistent with main model.
    """
    records = []
    for odds in odds_array:
        lr        = np.random.default_rng(seed)
        true_prob = min(1.0 / (odds * overround), 0.9999)
        wins      = lr.random(size=(n_sims, n_bets)) < true_prob

        balance    = np.full(n_sims, starting_balance, dtype=np.float64)
        cumul_loss = np.zeros(n_sims)
        peak_bal   = np.full(n_sims, starting_balance, dtype=np.float64)
        max_dd     = np.zeros(n_sims)
        max_stake  = np.zeros(n_sims)
        cur_loss   = np.zeros(n_sims, dtype=np.int32)
        max_loss   = np.zeros(n_sims, dtype=np.int32)
        busted     = np.zeros(n_sims, dtype=bool)
        npf        = (odds - 1.0) * (1.0 - commission)   # net profit per unit staked

        for i in range(n_bets):
            active = ~busted
            stake  = np.where(cumul_loss > 0, (cumul_loss + 1.0) / npf, 1.0)

            # Bust: required stake exceeds remaining balance
            busted |= (stake > balance) & active
            active  = ~busted
            stake   = np.where(active, stake, 0.0)
            max_stake = np.maximum(max_stake, stake)

            won  = wins[:, i] & active
            lost = ~wins[:, i] & active

            balance    = np.where(won,  balance + stake * npf, balance)
            balance    = np.where(lost, balance - stake,       balance)
            cumul_loss = np.where(won,  0.0,              cumul_loss)
            cumul_loss = np.where(lost, cumul_loss + stake, cumul_loss)
            cur_loss   = np.where(lost, cur_loss + 1, 0).astype(np.int32)
            max_loss   = np.maximum(max_loss, cur_loss)

            peak_bal = np.maximum(peak_bal, balance)
            max_dd   = np.maximum(max_dd, peak_bal - balance)

        records.append({
            'odds':              round(float(odds), 4),
            'win_pct':           round(true_prob * 100, 1),
            'bust_rate':         round(float(busted.mean() * 100), 1),
            'net_return_pct':    round(float((np.median(balance) / starting_balance - 1) * 100), 2),
            'median_final_bal':  round(float(np.median(balance)), 0),
            'p25_final_bal':     round(float(np.percentile(balance, 25)), 0),
            'p75_final_bal':     round(float(np.percentile(balance, 75)), 0),
            'median_max_dd':     round(float(np.median(max_dd)), 0),
            'p95_max_dd':        round(float(np.percentile(max_dd, 95)), 0),
            'median_max_loss':   round(float(np.median(max_loss)), 1),
            'p95_max_loss':      round(float(np.percentile(max_loss, 95)), 1),
            'median_max_stake':  round(float(np.median(max_stake)), 2),
            'p95_max_stake':     round(float(np.percentile(max_stake, 95)), 2),
        })
    return pd.DataFrame(records)


# Odds ranges to sweep
FAV_ODDS  = np.round(np.arange(1.01, 2.00, 0.05), 4)   # $1.01 – $1.96 in 5c steps
OUT_ODDS  = np.round(np.concatenate([
    np.arange(2.00, 5.00, 0.25),   # fine grain $2 – $5
    np.arange(5.00, 10.0, 0.50),   # medium grain $5 – $10
    np.arange(10.0, 16.0, 1.00),   # coarse $10 – $15
]), 4)

print(f'Favourites sweep: {len(FAV_ODDS)} odds levels  ({FAV_ODDS[0]:.2f} – {FAV_ODDS[-1]:.2f})')
print(f'Outsiders  sweep: {len(OUT_ODDS)} odds levels  ({OUT_ODDS[0]:.2f} – {OUT_ODDS[-1]:.2f})')
print(f'Each level: {SWEEP_N_SIMS} sims × {SWEEP_N_BETS:,} bets')

In [ ]:
import time

print('Running favourites sweep...')
t0 = time.time()
fav_sweep = sweep_fixed_odds(FAV_ODDS, seed=3000)
print(f'  done in {time.time()-t0:.1f}s')

print('Running outsiders sweep...')
t0 = time.time()
out_sweep = sweep_fixed_odds(OUT_ODDS, seed=4000)
print(f'  done in {time.time()-t0:.1f}s')

print('\nFavourites sweep — head:')
display(fav_sweep[['odds','win_pct','bust_rate','net_return_pct',
                    'median_max_dd','median_max_loss','median_max_stake']].head(8))
print('\nOutsiders sweep — head:')
display(out_sweep[['odds','win_pct','bust_rate','net_return_pct',
                   'median_max_dd','median_max_loss','median_max_stake']].head(8))

In [ ]:
# ── Viability score ────────────────────────────────────────────────────────
# Composite metric: rewards high survival and high capital retention.
#   score = (1 - bust_rate/100) × (median_final_bal / starting_balance)
# Range [0, 1]; higher = better balance of risk and return.

for df in (fav_sweep, out_sweep):
    df['viability'] = (1 - df['bust_rate'] / 100) * (df['median_final_bal'] / STARTING_BAL)


def find_sweet_spot(df):
    """Return the row with the highest viability score."""
    return df.loc[df['viability'].idxmax()]


fav_sweet = find_sweet_spot(fav_sweep)
out_sweet = find_sweet_spot(out_sweep)

print(f"Favourites sweet spot:  odds={fav_sweet['odds']:.2f}  "      f"win%={fav_sweet['win_pct']:.1f}  bust={fav_sweet['bust_rate']:.1f}%  "      f"viability={fav_sweet['viability']:.3f}")
print(f"Outsiders  sweet spot:  odds={out_sweet['odds']:.2f}  "      f"win%={out_sweet['win_pct']:.1f}  bust={out_sweet['bust_rate']:.1f}%  "      f"viability={out_sweet['viability']:.3f}")

In [ ]:
def plot_sweep(df, sweet_row, title, color, x_label='Odds'):
    """4-panel sweep chart: bust rate, net return, max drawdown, max loss streak."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle(title, fontsize=13, fontweight='bold')

    panels = [
        (axes[0, 0], 'bust_rate',        'Bust rate (%)',              False, 'lower'),
        (axes[0, 1], 'net_return_pct',   'Net return % (median)',      False, 'upper'),
        (axes[1, 0], 'median_max_dd',    'Median max drawdown ($)',     True,  'lower'),
        (axes[1, 1], 'median_max_loss',  'Median max loss streak',     False, 'lower'),
    ]

    for ax, col, ylabel, dollar, sweet_va in panels:
        ax.plot(df['odds'], df[col], color=color, lw=2)
        ax.fill_between(df['odds'], df[col], alpha=0.12, color=color)

        # p95 band where available
        p95_col = col.replace('median_', 'p95_').replace('bust_rate', '').replace('net_return_pct', '')
        if p95_col and p95_col in df.columns:
            ax.fill_between(df['odds'], df[col], df[p95_col],
                            alpha=0.08, color='grey', label='p95')

        # Sweet spot marker
        sv = sweet_row[col]
        ax.axvline(sweet_row['odds'], color='gold', ls='--', lw=1.5, alpha=0.9)
        ax.scatter([sweet_row['odds']], [sv], color='gold', s=80, zorder=5,
                   label=f"Sweet spot @ {sweet_row['odds']:.2f}")

        ax.set_xlabel(x_label)
        ax.set_ylabel(ylabel)
        if dollar:
            ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


plot_sweep(
    fav_sweep, fav_sweet,
    f'Favourites — Martingale metrics vs odds ($1.01 – $1.99)\n'
    f'{SWEEP_N_SIMS} sims × {SWEEP_N_BETS:,} bets  |  '
    f'Bankroll ${STARTING_BAL:,.0f}  |  Overround {OVERROUND*100:.0f}%  |  Commission {COMMISSION*100:.0f}%',
    FAVOURITE_COLOR,
)

plot_sweep(
    out_sweep, out_sweet,
    f'Outsiders — Martingale metrics vs odds ($2.00 – $15.00)\n'
    f'{SWEEP_N_SIMS} sims × {SWEEP_N_BETS:,} bets  |  '
    f'Bankroll ${STARTING_BAL:,.0f}  |  Overround {OVERROUND*100:.0f}%  |  Commission {COMMISSION*100:.0f}%',
    OUTSIDER_COLOR,
)

In [ ]:
# ── Composite viability curve — both agents overlaid ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, sweet, label, color in [
    (axes[0], fav_sweep, fav_sweet, 'Favourites ($1.01–$1.99)', FAVOURITE_COLOR),
    (axes[1], out_sweep, out_sweet, 'Outsiders  ($2.00–$15.00)', OUTSIDER_COLOR),
]:
    ax.plot(df['odds'], df['viability'], color=color, lw=2.5)
    ax.fill_between(df['odds'], df['viability'], alpha=0.15, color=color)
    ax.axvline(sweet['odds'], color='gold', ls='--', lw=2, label=f"Sweet spot: {sweet['odds']:.2f}")
    ax.scatter([sweet['odds']], [sweet['viability']], color='gold', s=100, zorder=5)
    ax.annotate(
        f" odds={sweet['odds']:.2f}\n win%={sweet['win_pct']:.1f}%\n bust={sweet['bust_rate']:.1f}%",
        xy=(sweet['odds'], sweet['viability']),
        xytext=(10, -30), textcoords='offset points',
        fontsize=8, color='black',
        arrowprops=dict(arrowstyle='->', color='grey', lw=1),
    )
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Odds')
    ax.set_ylabel('Viability score  (survival × capital retention)')
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle(
    'Composite Viability Score = (1 – bust rate) × (median final balance / starting balance)\n'
    'Higher = better balance of survival probability and capital preservation',
    fontsize=11, fontweight='bold',
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Sweet spot summary tables ─────────────────────────────────────────────
print('=== TOP 5 FAVOURITE ODDS BY VIABILITY ===')
display(
    fav_sweep.nlargest(5, 'viability')[[
        'odds', 'win_pct', 'bust_rate', 'net_return_pct',
        'median_max_dd', 'median_max_loss', 'median_max_stake', 'viability'
    ]].rename(columns={
        'odds': 'Odds', 'win_pct': 'Win %', 'bust_rate': 'Bust %',
        'net_return_pct': 'Net Return %', 'median_max_dd': 'Med Max DD ($)',
        'median_max_loss': 'Med Max Loss Streak', 'median_max_stake': 'Med Max Stake ($)',
        'viability': 'Viability Score',
    }).reset_index(drop=True)
)

print('\n=== TOP 5 OUTSIDER ODDS BY VIABILITY ===')
display(
    out_sweep.nlargest(5, 'viability')[[
        'odds', 'win_pct', 'bust_rate', 'net_return_pct',
        'median_max_dd', 'median_max_loss', 'median_max_stake', 'viability'
    ]].rename(columns={
        'odds': 'Odds', 'win_pct': 'Win %', 'bust_rate': 'Bust %',
        'net_return_pct': 'Net Return %', 'median_max_dd': 'Med Max DD ($)',
        'median_max_loss': 'Med Max Loss Streak', 'median_max_stake': 'Med Max Stake ($)',
        'viability': 'Viability Score',
    }).reset_index(drop=True)
)

print(f"\n{'─'*60}")
print(f"  SUMMARY")
print(f"{'─'*60}")
print(f"  Favourites sweet spot : ${fav_sweet['odds']:.2f}  "      f"(win rate {fav_sweet['win_pct']:.1f}%, bust {fav_sweet['bust_rate']:.1f}%)")
print(f"  Outsiders  sweet spot : ${out_sweet['odds']:.2f}  "      f"(win rate {out_sweet['win_pct']:.1f}%, bust {out_sweet['bust_rate']:.1f}%)")
print(f"{'─'*60}")
print()
print("  Key insight: very short odds (e.g. $1.05) appear 'safe' due to high win")
print("  rates, but Martingale stake escalation per loss is inversely proportional")
print("  to the net profit margin — making short-odds the most dangerous bracket.")